In [ ]:
import os
import matplotlib.pyplot as plt
import re
import gzip
import shutil
import pandas as pd
import scipy.sparse
from multiprocessing import Pool
import tarfile
import seaborn as sns
import scanpy.external as sce 
import harmonypy as hm
import anndata as ad
import omicverse as ov
import scvi
import scib
import annoy
import gc
import numpy as np
import random
import scanpy as sc

In [ ]:
sc.settings.verbosity = 1 
sc.logging.print_versions()
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(4, 4), facecolor='white')
ov.style(font_path='arial')

In [ ]:
os.chdir("")
os.getcwd()

In [ ]:
def save_fig(
    fig_name,
    save_dir="",
    fmt="pdf",
    dpi=300,
):
    save_fig.counter = getattr(save_fig, "counter", 1)
    filename = f"{save_fig.counter:02d}_{fig_name}.{fmt}"
    plt.savefig(
        os.path.join(save_dir, filename),
        format=fmt,
        dpi=dpi,
        bbox_inches="tight",
    )
    save_fig.counter += 1

In [ ]:
adata = ov.read("/20260911Allo_Xeno_PBMC.h5ad")

In [ ]:
adata

In [ ]:
drop_cols = [
    "celltypist_high",
    "celltypist_low",
    "celltype_coarse",
    "celltype_coarse_1",
    "celltype_final",
]
adata.obs.drop(columns=[col for col in drop_cols if col in adata.obs.columns], inplace=True)

rename_dict = {
    "celltype_coarse_2": "celltype_coarse_1",
    "celltype_final_1": "celltype_final",
}
adata.obs.rename(columns=rename_dict, inplace=True)

In [ ]:
ov.pl.umap(
    adata,
    color=["celltype_final",'celltype_coarse_1','leiden_auto',"batch","orig.ident","species","sample_1","sample_3","source"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig('figure')
plt.show()

Monocytes

In [ ]:
adata_mono = adata[
    adata.obs["celltype_coarse_1"].str.contains("Monocytes", case=False, na=False)
].copy()

In [ ]:
adata_mono.X = adata_mono.layers['counts'].copy()

In [ ]:
adata_mono

In [ ]:
adata_mono.X.max()

In [ ]:
adata_mono.var_names_make_unique()
adata_mono.obs_names_make_unique()

In [ ]:
adata_mono

In [ ]:
sc.pp.normalize_total(adata_mono, target_sum=1e4)
sc.pp.log1p(adata_mono)
adata_mono.layers['lognorm'] = adata_mono.X.copy()
adata_mono.raw = adata_mono.copy()                           

In [ ]:
sc.pp.highly_variable_genes(
    adata_mono, flavor='seurat', n_top_genes=2000,
    batch_key='orig.ident', subset=False,
)

In [ ]:
sc.pp.scale(adata_mono, max_value=10)
sc.tl.pca(adata_mono, n_comps=50, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_mono, n_pcs=50)

In [ ]:
import scanpy.external as sce
sce.pp.harmony_integrate(
    adata_mono, key='batch', basis='X_pca', adjusted_basis='X_pca_harmony',
    theta=2, sigma=0.1, max_iter_harmony=30,
)

In [ ]:
sc.pp.neighbors(adata_mono, n_neighbors=10, use_rep='X_pca_harmony', n_pcs=10)
sc.tl.umap(adata_mono)

In [ ]:
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]
res_keys = []
for res in resolutions:
    res_keys.append(f"leiden_res_{res:.1f}")
for res, key in zip(resolutions, res_keys):
    sc.tl.leiden(
        adata_mono, 
        resolution=res, 
        flavor="igraph", 
        n_iterations=-1, 
        key_added=key
    )

In [ ]:
sc.pl.embedding(
    adata_mono,
    basis="UMAP", 
    color=res_keys,
    ncols=3,
    size=0.8,
    frameon=False,
    wspace=0.2,
    hspace=0.3,
    show=False
)
save_fig("UMAP亚群")
plt.show()

In [ ]:
adata_mono.obs['leiden_auto'] = adata_mono.obs['leiden_res_0.6'].astype(str)

In [ ]:
ov.pl.umap(
    adata_mono,
    color=["batch","phase",'leiden_auto',"source","orig.ident","species","sample_1","sample_3"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig("UMAP亚群")
plt.show()

In [ ]:
with plt.rc_context({'figure.figsize': (20, 4), 'figure.dpi': 300}):
    for metric in ['n_genes_by_counts', 'total_counts',
                    'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']:
        sc.pl.violin(adata_mono, metric, jitter=0, groupby='leiden_auto',
                    rotation=90, show=False)
        save_fig('figure')
        plt.show()

In [ ]:
adata_mono.obs['orig.ident'].value_counts()

In [ ]:
adata_mono.obs['leiden_auto'].value_counts()

In [ ]:
# 暂时屏蔽线粒体、核糖体和血红蛋白基因
gene_names = adata_mono.var_names.str.upper()

adata_mono.var["use_for_marker"] = ~(
    gene_names.str.startswith("MT-") |
    gene_names.str.startswith("RPL") |
    gene_names.str.startswith("RPS") |
    gene_names.isin(["HBA1", "HBA2", "HBB", "HBD"])
)

In [ ]:
sc.tl.rank_genes_groups(
    adata_mono,
    groupby="leiden_auto",
    groups="all",
    reference="rest",
    layer="lognorm",
    use_raw=False,
    method="wilcoxon",
    pts=True,
    n_genes=100,
    mask_var="use_for_marker"
)


In [ ]:
sc.tl.dendrogram(
    adata_mono,
    groupby="leiden_auto"
)

sc.pl.rank_genes_groups_dotplot(
    adata_mono,
    groupby="leiden_auto",
    n_genes=3,
    layer="lognorm",
    use_raw=False,
    standard_scale="var",
    cmap="RdBu_r",
    show=False
)
plt.show()

In [ ]:
cluster_counts = adata_mono.obs["leiden_auto"].value_counts()
cluster_counts

In [ ]:
top_20_genes = pd.DataFrame(
    adata_mono.uns["rank_genes_groups"]["names"]
).head(20)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print(top_20_genes.T.to_string())

Figure 2A

In [ ]:
mono_map = {
    "0": "FCGR3A+ Mono",
    "1": "HLA-DRA+ Mono",
    "2": "CD14+ Mono",
    "3": "VCAN+ Mono",
    "4": "PF4+PPBP+ Mono",
    "5": "VCAN+ Mono",
    "6": "S100A12+ Mono",
    "7": "S100A12+ Mono",
    "8": "S100A12+ Mono",
}

In [ ]:

adata_mono.obs["mono_subtype"] = (
    adata_mono.obs["leiden_auto"]
    .astype(str)
    .map(mono_map)
)

In [ ]:
ov.pl.umap(
    adata_mono,
    color=['mono_subtype',"batch","phase",'leiden_auto',"source","orig.ident","species","sample_1","sample_3"],
    frameon='small',
    palette=ov.pl.sc_color,
    ncols=1,
    show=False
)
save_fig("UMAP亚群")
plt.show()

Figure 1B

In [ ]:
marker_panel = {
    'CD14+ Mono':      ['CD14'],                        
    'FCGR3A+ Mono':    ['FCGR3A', 'CDKN1C', 'MS4A7'],
    'HLA-DRA+ Mono':   ['HLA-DRA', 'HLA-DMA', 'CPVL'],
    'PF4+PPBP+ Mono':  ['PPBP', 'PF4', 'GP1BB'],
    'S100A12+ Mono':   ['S100A12', 'ALOX5AP', 'PADI4'],
    'VCAN+ Mono':      ['VCAN', 'MS4A4E', 'PDE4D'],
}

In [ ]:

sc.pl.dotplot(
    adata_mono,
    var_names=marker_panel,
    groupby="mono_subtype",
    layer="lognorm",
    use_raw=False,
    standard_scale="var",
    cmap="RdBu_r",
    dendrogram=True,
    show=False,
)
save_fig("Dotplot_亚群")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 5))
ov.pl.cellproportion(adata=adata_mono, celltype_clusters='mono_subtype', groupby='source', legend=False, ax=ax, show=False)
ax.legend(*ax.get_legend_handles_labels(), bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
save_fig('figure')
plt.show()

Figure 2D

In [ ]:
marker_check = {
    "Monocyte Identity": [
        "LST1",
        "TYROBP",
        "FCER1G",
        "CTSS",
        "LILRB1",
        "CTSD",
    ],
    "Platelet Program": [
        "PPBP",
        "PF4",
        "NRGN",
        "GNG11",
        "RGS18",
        "CLU",
        "CAVIN2",
    ],
    "Myeloid Activation": [
        "S100A8",
        "S100A9",
        "S100A12",
        "FCGR3A",
        "HLA-DRA",
    ],
    "Lineage Exclusion": ["CD3D", "MS4A1", "HBA1"],
}


In [ ]:
sc.pl.dotplot(
    adata_mono,
    marker_check,
    groupby="leiden_auto",
    layer="lognorm",
    use_raw=False,
    standard_scale="var",
    dendrogram=False,
    show=False
)
plt.show()

scCODA

Figure 2E

In [ ]:
adata_mono_allo = adata_mono[
    adata_mono.obs["orig.ident"] == "OMIX012347"
].copy()


In [ ]:

adata_mono_xeno = adata_mono[
    adata_mono.obs["orig.ident"] == "Xen_17390399"
].copy()

In [ ]:
adata_mono_allo.obs["condition"] = pd.Categorical(
    adata_mono_allo.obs["condition"],
    categories=["Normal", "AMR"],
    ordered=True,
)


In [ ]:
dct_obj = ov.single.DCT(
    adata=adata_mono_allo,
    condition="condition",
    ctrl_group="Normal",
    test_group="AMR",
    cell_type_key="mono_subtype",
    method="sccoda",
    sample_key="batch",
)


In [ ]:
dct_obj.sccoda_data = dct_obj.model.prepare(
    dct_obj.sccoda_data,
    modality_key="coda",
    formula="condition",
    reference_cell_type="automatic",
)


In [ ]:
ref_cell_allo = dct_obj.sccoda_data["coda"].uns["scCODA_params"][
    "reference_cell_type"
]
print(f"Reference cell type: {ref_cell_allo}")

In [ ]:
dct_obj.run(num_samples=5000, num_warmup=500, rng_key=42)
dct_obj.model.set_fdr(dct_obj.sccoda_data, modality_key="coda", est_fdr=0.1)
res_allo = dct_obj.get_results()
print("=== Allo scCODA Results ===")
print(f"Reference cell type: {ref_cell_allo}")
display(res_allo.sort_values("Inclusion probability", ascending=False))
plt.show()

In [ ]:
ov.plot_set()
dct_obj.model.plot_boxplots(
    dct_obj.sccoda_data,
    modality_key="coda",
    feature_name="condition",
    add_dots=True,
    figsize=(8, 5),
    dpi=300,
)

# 调整布局
save_fig("Allo_scCODA_Boxplots", fmt="pdf", dpi=300)
ov.plt.show()

In [ ]:
import logging
import pertpy as pt

day_str = adata_mono_xeno.obs["batch"].astype(str).str.split("_").str[0]
adata_mono_xeno.obs["day_raw"] = day_str.str.replace(
    "^D", "", regex=True
).astype(float)

min_d = adata_mono_xeno.obs["day_raw"].min()
max_d = adata_mono_xeno.obs["day_raw"].max()
adata_mono_xeno.obs["day_norm"] = (adata_mono_xeno.obs["day_raw"] - min_d) / (
    max_d - min_d
)


In [ ]:
sccoda_model = pt.tl.Sccoda()
sccoda_data = sccoda_model.load(
    adata_mono_xeno,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="mono_subtype",
    sample_identifier="batch",
    covariate_obs=["day_norm"],
)


In [ ]:
sccoda_data = sccoda_model.prepare(
    sccoda_data,
    modality_key="coda",
    formula="day_norm",
    reference_cell_type="automatic",
)
ref_cell_xeno = sccoda_data["coda"].uns["scCODA_params"]["reference_cell_type"]


In [ ]:
res = sccoda_model.summary(sccoda_data, modality_key="coda")
print(res)

In [ ]:
sccoda_model.run_nuts(
    sccoda_data,
    modality_key="coda",
    num_samples=5000,
    num_warmup=500,
    rng_key=42,
)
sccoda_model.set_fdr(sccoda_data, modality_key="coda", est_fdr=0.1)


In [ ]:
ref_cell = sccoda_data["coda"].uns["scCODA_params"]["reference_cell_type"]
print(f"当前模型 Reference Cell Type: {ref_cell}")


In [ ]:

effect_df = sccoda_model.get_effect_df(
    sccoda_data, modality_key="coda"
).reset_index()

cov_col = [c for c in effect_df.columns if "covariate" in c.lower()][0]
ct_col = [
    c for c in effect_df.columns if "cell" in c.lower() and "type" in c.lower()
][0]
eff_time = effect_df[
    effect_df[cov_col].astype(str).str.contains("day_norm")
].copy()

lfc_cols = [
    c
    for c in eff_time.columns
    if "log2" in c.lower() or "fold" in c.lower() or "lfc" in c.lower()
]
lfc_col = lfc_cols[0] if lfc_cols else None

cred_cols = [
    c
    for c in eff_time.columns
    if "credible" in c.lower() or "is_cred" in c.lower()
]
cred_col = cred_cols[0] if cred_cols else None

auto_label_map = {}

for _, row in eff_time.iterrows():
    ct = row[ct_col]

    if ct == ref_cell:
        auto_label_map[ct] = f"{ct} (Reference)"
        continue

    is_cred = bool(row[cred_col]) if cred_col else False
    sig_str = "*" if is_cred else "ns"

    lfc_val = row[lfc_col] if (lfc_col and pd.notna(row[lfc_col])) else None

    if lfc_val is not None:
        auto_label_map[ct] = f"{ct} {sig_str} (log2FC = {float(lfc_val):+.2f})"
    else:
        auto_label_map[ct] = f"{ct} {sig_str}"

for k, v in auto_label_map.items():
    print(f"  {k} -> {v}")


In [ ]:
coda_counts = pd.DataFrame(
    sccoda_data["coda"].X,
    index=sccoda_data["coda"].obs_names,
    columns=sccoda_data["coda"].var_names,
)
coda_props = coda_counts.div(coda_counts.sum(axis=1), axis=0) * 100

batch_names = sccoda_data["coda"].obs["batch"].astype(str)
coda_props["day"] = (
    batch_names.str.extract(r"^D(-?\d+(?:\.\d+)?)")[0].astype(float).values
)
plot_df = coda_props.sort_values("day").drop(columns=["day"])

bar_colors = [color_dict.get(col, "#888888") for col in plot_df.columns]

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=300)

plot_df.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=bar_colors,
    width=0.75,
    edgecolor="none",
)

handles, labels = ax.get_legend_handles_labels()
new_labels = [auto_label_map.get(l, l) for l in labels]

ax.legend(
    handles,
    new_labels,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=8.5,
    title="scCODA Dynamics\n(* Credible at FDR < 0.1)",
    title_fontsize=9,
)

ax.set_ylabel("Proportion in Monocytes (%)", fontsize=10)
ax.set_xlabel("Longitudinal Samples (Chronological POD)", fontsize=10)
ax.set_xticklabels(plot_df.index, rotation=45, ha="right", fontsize=8.5)
ax.set_title(
    "Longitudinal Monocyte Subset Composition (Xenotransplantation)",
    fontsize=11,
    fontweight="bold",
    pad=10,
)
ax.set_ylim(0, 100)
sns.despine(ax=ax, top=True, right=True)

plt.tight_layout()
plt.savefig(
    "Xeno_scCODA_longitudinal_stacked_barplot.pdf", dpi=300, bbox_inches="tight"
)
plt.show()

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if "mono_subtype_colors" in adata_mono.uns:
    MONO_SUBTYPES = list(adata_mono.obs["mono_subtype"].cat.categories)
    MONO_COLORS = list(adata_mono.uns["mono_subtype_colors"])
    MONO_PALETTE = dict(zip(MONO_SUBTYPES, MONO_COLORS))
else:
    MONO_SUBTYPES = list(adata_mono.obs["mono_subtype"].unique())
    import seaborn as sns

    MONO_COLORS = sns.color_palette("tab10", n_colors=len(MONO_SUBTYPES)).as_hex()
    MONO_PALETTE = dict(zip(MONO_SUBTYPES, MONO_COLORS))

fig = plt.figure(figsize=(13.5, 9), dpi=300)
gs = gridspec.GridSpec(
    2,
    2,
    width_ratios=[1.2, 0.8],
    height_ratios=[1, 1.1],
    hspace=0.35,
    wspace=0.28,
)

ax_a = plt.subplot(gs[0, 0])
# 确保列按统一的亚群排序
allo_cols = [c for c in MONO_SUBTYPES if c in allo_plot_df.columns]
allo_plot_df[allo_cols].plot(
    kind="bar",
    stacked=True,
    ax=ax_a,
    color=[MONO_PALETTE[c] for c in allo_cols],
    width=0.45,
    edgecolor="none",
    legend=False,
)
ax_a.set_title(
    "A   Allotransplantation: Sample Composition",
    loc="left",
    fontsize=11,
    fontweight="bold",
)
ax_a.set_ylabel("Proportion (%)", fontsize=9.5)
ax_a.set_xlabel("")
ax_a.set_ylim(0, 100)
ax_a.set_xticklabels(allo_plot_df.index, rotation=25, ha="right", fontsize=9)
ax_a.spines[["top", "right"]].set_visible(False)

ax_b = plt.subplot(gs[0, 1])

res_allo_df = res_allo.copy()
ct_col_a = [
    c for c in res_allo_df.columns if "cell" in c.lower() and "type" in c.lower()
][0]
res_allo_df = res_allo_df.rename(columns={ct_col_a: "Cell_Type"})
res_allo_df = res_allo_df[res_allo_df["Cell_Type"].isin(MONO_SUBTYPES)].copy()
res_allo_df["Cell_Type"] = pd.Categorical(
    res_allo_df["Cell_Type"], categories=MONO_SUBTYPES, ordered=True
)
res_allo_df = res_allo_df.sort_values("Cell_Type").reset_index(drop=True)
res_allo_df["y_pos"] = range(len(res_allo_df))

for _, row in res_allo_df.iterrows():
    ct = row["Cell_Type"]
    y = row["y_pos"]

    final_param = float(row.get("Final Parameter", 0.0))
    lfc = float(
        row.get(
            "log2-fold change", row.get("log2_fold_change", final_param)
        )
    )

    is_cred = abs(final_param) > 1e-4

    ax_b.hlines(
        y=y,
        xmin=0,
        xmax=lfc if ct != ref_allo else 0,
        color="#888888",
        lw=1.2,
        zorder=1,
    )

    if ct == ref_allo:
        ax_b.scatter(0, y, color="#555555", s=55, marker="D", zorder=3)
        ax_b.text(
            0.06,
            y,
            "Reference",
            va="center",
            ha="left",
            fontsize=7.5,
            color="#555555",
            fontstyle="italic",
        )
    elif is_cred:
        ax_b.scatter(
            lfc,
            y,
            color="#C14B3A",
            edgecolors="black",
            s=70,
            zorder=3,
        )
    else:
        ax_b.scatter(
            lfc,
            y,
            facecolors="none",
            edgecolors="#777777",
            s=50,
            linewidth=1.2,
            zorder=3,
        )

ax_b.axvline(0, color="gray", linestyle="--", lw=0.8, zorder=0)
ax_b.set_yticks(res_allo_df["y_pos"])
ax_b.set_yticklabels(res_allo_df["Cell_Type"], fontsize=9)
ax_b.set_xlabel("Effect size (log2FC vs Normal)", fontsize=9)
ax_b.set_title(
    "B   Allotransplantation: scCODA Effects",
    loc="left",
    fontsize=11,
    fontweight="bold",
)
ax_b.spines[["top", "right"]].set_visible(False)

ax_c = plt.subplot(gs[1, 0])
xeno_cols = [c for c in MONO_SUBTYPES if c in xeno_plot_df.columns]
xeno_plot_df[xeno_cols].plot(
    kind="bar",
    stacked=True,
    ax=ax_c,
    color=[MONO_PALETTE[c] for c in xeno_cols],
    width=0.75,
    edgecolor="none",
    legend=False,
)
ax_c.set_title(
    "C   Xenotransplantation: Longitudinal Dynamics",
    loc="left",
    fontsize=11,
    fontweight="bold",
)
ax_c.set_ylabel("Proportion (%)", fontsize=9.5)
ax_c.set_xlabel("Time post-transplantation (POD)", fontsize=9.5)
ax_c.set_ylim(0, 100)
ax_c.set_xticklabels(xeno_plot_df.index, rotation=45, ha="right", fontsize=8)
ax_c.spines[["top", "right"]].set_visible(False)

ax_d = plt.subplot(gs[1, 1])

cov_col = [c for c in res_xeno.columns if "covariate" in c.lower()][0]
ct_col_x = [
    c for c in res_xeno.columns if "cell" in c.lower() and "type" in c.lower()
][0]

res_xeno_df = res_xeno[
    res_xeno[cov_col].astype(str).str.contains("day_norm")
].copy()
res_xeno_df = res_xeno_df.rename(columns={ct_col_x: "Cell_Type"})
res_xeno_df = res_xeno_df[res_xeno_df["Cell_Type"].isin(MONO_SUBTYPES)].copy()

res_xeno_df["Cell_Type"] = pd.Categorical(
    res_xeno_df["Cell_Type"], categories=MONO_SUBTYPES, ordered=True
)
res_xeno_df = res_xeno_df.sort_values("Cell_Type").reset_index(drop=True)
res_xeno_df["y_pos"] = range(len(res_xeno_df))

for _, row in res_xeno_df.iterrows():
    ct = row["Cell_Type"]
    y = row["y_pos"]

    final_param = float(row.get("Final Parameter", 0.0))
    lfc = float(
        row.get(
            "log2-fold change", row.get("log2_fold_change", final_param)
        )
    )

    is_cred = abs(final_param) > 1e-4

    ax_d.hlines(
        y=y,
        xmin=0,
        xmax=lfc if ct != ref_xeno else 0,
        color="#888888",
        lw=1.2,
        zorder=1,
    )

    if ct == ref_xeno:
        ax_d.scatter(0, y, color="#555555", s=55, marker="D", zorder=3)
        ax_d.text(
            0.06,
            y,
            "Reference",
            va="center",
            ha="left",
            fontsize=7.5,
            color="#555555",
            fontstyle="italic",
        )
    elif is_cred:
        ax_d.scatter(
            lfc,
            y,
            color="#C14B3A",
            edgecolors="black",
            s=70,
            zorder=3,
        )
    else:
        ax_d.scatter(
            lfc,
            y,
            facecolors="none",
            edgecolors="#777777",
            s=50,
            linewidth=1.2,
            zorder=3,
        )

ax_d.axvline(0, color="gray", linestyle="--", lw=0.8, zorder=0)
ax_d.set_yticks(res_xeno_df["y_pos"])
ax_d.set_yticklabels(res_xeno_df["Cell_Type"], fontsize=9)
ax_d.set_xlabel("Time-association effect (log2FC)", fontsize=9)
ax_d.set_title(
    "D   Xenotransplantation: scCODA Time Effects",
    loc="left",
    fontsize=11,
    fontweight="bold",
)
ax_d.spines[["top", "right"]].set_visible(False)

color_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="s",
        color="w",
        markerfacecolor=MONO_PALETTE[c],
        markersize=8.5,
        label=c,
    )
    for c in MONO_SUBTYPES
]

stat_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="#C14B3A",
        markeredgecolor="black",
        markersize=7.5,
        label="Credible (FDR < 0.1)",
    ),
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="none",
        markeredgecolor="#777777",
        markersize=6.5,
        label="Non-credible",
    ),
    plt.Line2D(
        [0],
        [0],
        marker="D",
        color="w",
        markerfacecolor="#555555",
        markersize=6.5,
        label="Reference baseline",
    ),
]

fig.legend(
    handles=color_handles + stat_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.06),
    ncol=5,
    frameon=False,
    fontsize=8.5,
)

plt.savefig(
    "Fig2_Monocyte_scCODA_Integrated_2x2.pdf", dpi=300, bbox_inches="tight"
)
plt.show()

Figure 2C

In [ ]:
import pyucell as uc

signatures = {
    "Platelet_Associated": [
        "GP9",
        "GP1BA",
        "GP1BB",
        "TUBB1",
        "RGS18",
        "ITGA2B",
        "ITGB3",
        "TREML1",
        "MPIG6B",
        "NRGN",
        "SDPR",
        "CLU",
    ],
    "Inflammatory_Activation": [
        "S100A8",
        "S100A9",
        "S100A12",
        "IL1A",
        "IL1B",
        "TNF",
        "CXCL8",
        "CCL2",
    ],
}

# 设为 n_jobs=1，避开多进程反序列化问题
uc.compute_ucell_scores(
    adata_mono,
    signatures=signatures,
    layer="lognorm",
    missing_genes="skip",
    n_jobs=1,
)

# 打印结果检查
score_cols = ["Platelet_Associated_UCell", "Inflammatory_Activation_UCell"]
display(adata_mono.obs.groupby("mono_subtype")[score_cols].mean().round(4))

In [ ]:
# 提取刚才计算好的两个 UCell 得分列名
score_metrics = ["Platelet_Associated_UCell", "Inflammatory_Activation_UCell"]

with plt.rc_context({"figure.figsize": (8, 4), "figure.dpi": 300}):
    for metric in score_metrics:
        sc.pl.violin(
            adata_mono,
            keys=metric,
            groupby="mono_subtype",
            jitter=0,
            rotation=45,
            show=False,
        )
        plt.title(f"{metric} across Monocyte Subtypes", fontsize=12, pad=10)
        plt.tight_layout()

        save_fig("Ucells评分violin")  # 你的保存函数
        send_plot()  # 你的发送函数
        plt.show()